# Chapter 17 — Micro LLM Configuration

This is the capstone of the technical curriculum: build a complete micro LLM with a chat interface that reflects your personality. You'll combine every technique from the course — tokenizer, transformer, training, fine-tuning, alignment, and deployment — into a single cohesive project.

### Glossary / Glossário

• micro LLM — small language model (typically under 1B parameters) designed for constrained environments. / Modelo de linguagem pequeno (tipicamente menos de 1B parâmetros) para ambientes restritos.
• d_model — model dimension, the size of the hidden state vector throughout the transformer. / Dimensão do modelo, tamanho do vetor de estado oculto ao longo do transformer.
• n_heads — number of attention heads in multi-head attention. / Número de cabeças de atenção no multi-head attention.
• d_ff — feed-forward dimension, typically 4× d_model. / Dimensão feed-forward, tipicamente 4× d_model.
• decoder-only — transformer variant using only masked self-attention, standard for language model generation. / Variante do transformer que usa apenas self-attention mascarada, padrão para geração de modelos de linguagem.

In [ ]:
from dataclasses import dataclass

@dataclass
class MicroLLMConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_heads: int = 12
    n_layers: int = 12
    d_ff: int = 3072
    max_seq_len: int = 1024
    dropout: float = 0.1

# Calculate total parameters
c = MicroLLMConfig()
embed_params = c.vocab_size * c.d_model
attn_params = 4 * c.d_model * c.d_model  # Q, K, V, O projections
ffn_params = 2 * c.d_model * c.d_ff      # up + down
layer_params = attn_params + ffn_params
total = embed_params + (c.n_layers * layer_params) + (c.vocab_size * c.d_model)
print(f"Embedding:  {embed_params:>12,}")
print(f"Per layer:  {layer_params:>12,}")
print(f"All layers: {c.n_layers * layer_params:>12,}")
print(f"LM head:    {c.vocab_size * c.d_model:>12,}")
print(f"Total:      {total:>12,} (~{total/1e6:.0f}M parameters)")

# === Aha: how d_model affects total parameters ===
print(f"\n{'d_model':>8s} | {'n_heads':>8s} | {'Total Params':>14s} | {'Size (FP16)':>12s}")
print("-" * 50)
for dm in [256, 512, 768, 1024, 2048]:
    nh = dm // 64  # one head per 64 dims
    nl = 12
    dff = dm * 4
    ep = 32000 * dm
    ap = 4 * dm * dm
    fp = 2 * dm * dff
    lp = ap + fp
    t = ep + nl * lp + 32000 * dm
    size_mb = t * 2 / 1e6  # FP16 = 2 bytes per param
    print(f"{dm:>8d} | {nh:>8d} | {t:>14,} | {size_mb:>10.1f} MB")
print("\nDoubling d_model roughly 4x the parameters — scaling is quadratic!")

---

**Course:** Aprenda LLM | **Chapter 17**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.